<a href="https://colab.research.google.com/github/stuart7824/Pandas/blob/main/WACC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fredapi

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
# WACC = (Weight of Equity * Cost of Equity) + (Weight of Debt * PreTax Cost of Debt * (1- Marginal Tax Rate))



*   The wacc calculator is ONLY for Indian stocks, cause for CAPM it uses the returns of nifty 50 index
*   By default the stock price will be fetched from NSE. IF the company is not on NSE ONLY THEN, the stock price will be fetched from BSE
*   If the company name entered is incorrectly then please try again



## Weights

In [4]:
#enter the stock symbol from yahoo finance
company=str(input("Which Indian companany's WACC do you want to calculate? : ")).upper()
nse,bse=".NS",".BO"
df=company+nse
a=yf.Ticker(df)

#----------------------------------------------------------------------------
# Check if data exists by fetching a small piece of info
#/**if 'regularMarketPrice' not in a.info and 'currentPrice' not in a.info:
#    # 2. If NSE fails, switch to .BO (BSE)
#    print(f"{df} not found on NSE, trying BSE...")
#    ticker_bo = company + ".BO"
#    a = yf.Ticker(ticker_bo)

    # 3. Check if BSE also fails
#    if 'regularMarketPrice' not in a.info and 'currentPrice' not in a.info:
#        print("Error: Company not found on either NSE or BSE.")
#    else:
#        print(f"Success: Using {ticker_bo}")
#else:
 #   print(f"Success: Using {df}")**/
#---------------------------------------------------------------------------

Which Indian companany's WACC do you want to calculate? : itc


In [5]:
if pd.isna(a.info.get('previousClose') or a.info.get('sharesOutstanding')):
  print("COULDN'T FETCH DETAILS!")
else:
  equity=(a.info['previousClose']*a.info['sharesOutstanding'])
print("equity value: ₹",equity)

equity value: ₹ 3647954675455.65


In [6]:
if pd.isna(a.info.get('totalDebt')):
    debt=0
    print("COULDN'T FETCH THE TOTAL DEBT!")
else:
    debt = float(a.info['totalDebt'])
    print(f"debt value: ₹ {debt}")

debt value: ₹ 23990599680.0


In [7]:
marketcap=debt+equity
print("market value: ₹",marketcap)

market value: ₹ 3671945275135.65


In [8]:
c=equity/marketcap
d=debt/marketcap
print("Equity weight:",round((c*100),2),"%")
print("Debt weight:", round(d*100,2),"%")

Equity weight: 99.35 %
Debt weight: 0.65 %


# Cost of equity

In [9]:
#CAPM (Capital Asset Pricing Model)

In [10]:
#Risk-Free Rate sourced from FRED as of 18/06/2026

In [11]:
from fredapi import Fred
fred=Fred(api_key='921d01c28bf1cb926d7623b162cf9009')
riskfreerate=fred.get_series('INDIRLTLT01STM')


# Instead of just printing the tail, store the tail
last_entry = riskfreerate.tail(1)

# .iloc[0] grabs the value at the first (and only) position
riskfree = (last_entry.iloc[0])/100

print(riskfree)

0.0702


In [12]:
#beta
beta=a.info['beta']
print(beta/100)

-0.0008


In [13]:
#mrp = market risk premium / erp = equity risk premium
nifty=0.125
mrp=nifty-riskfree
print(mrp)

0.0548


In [14]:
#capm=riskfreerate+beta(mrp)
capm=riskfree+beta*(mrp)
print(capm)

0.065816


# Cost of debt

In [15]:
#pretax cost of debt
interest=a.income_stmt.loc['Interest Expense'].iloc[0]
pretax=interest/debt
print(pretax)

0.03550140519038497


In [16]:
#marginal tax rate
t=0.3
print(t)

0.3


# **WACC**

In [18]:
wacc=round(((c*capm)+(d*pretax*(1-t)))*100,2)
print("The WACC for",company,"is",wacc,"%")

The WACC for ITC is 6.55 %
